# Session 08: Pipelines and platform design

Build the three step pipeline for the Kitara mobile money log, break it with a renamed column, and fill the three zones with SQL. Then draw your own pipeline map on the page Pipeline map of the Participant Workbook.

**Student version.** This is the team notebook. Leaders can follow it on the shared screen. Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("customers.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith(".csv")), "and the docs folder")

## Setup: read the mobile money log
This cell reads `mobile_money_log.csv` as text. Run it once. A month of agent transactions, 20,000 rows, with duplicates, two date formats and one long silence.

In [ ]:
import pandas as pd, sqlite3, tempfile, pathlib
raw = pd.read_csv('mobile_money_log.csv', dtype=str)
print(len(raw), 'rows,', list(raw.columns))

## Task 1. One date format
Two date formats sit in the `timestamp` column: `2026-09-01 00:06:29` and `01/09/2026 04:56` (day first). Write `parse_times(df)` that returns the table with a new column `ts` holding real dates. Try the first format, then the second for the rows that failed. Store the result in `log`.

In [ ]:
def parse_times(df):
    # Your turn: iso = pd.to_datetime(df.timestamp, errors='coerce', format='%Y-%m-%d %H:%M:%S')
    # Your turn: dmy = the same with format='%d/%m/%Y %H:%M'
    # Your turn: return df.assign(ts=...) using iso and fillna with dmy
    pass

log = parse_times(raw)


In [ ]:
_missing = [v for v in ('log',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert log.ts.notna().all(), 'Every row should have a time. Try the second format where the first fails (fillna)'
assert str(log.ts.min()) == '2026-09-01 00:06:29', 'The first transaction is at 2026-09-01 00:06:29'
assert raw.timestamp.str.contains('/').sum() == 1842, 'There are 1,842 rows in the slash format'
print('Task 1 correct')

## Task 2. Duplicates, freshness and volume
Count the rows that repeat an earlier `txn_id` (keep the first copy) in `n_dup`. Find the longest silence between two transactions, in hours, in `gap_hours` (use `log.ts.sort_values().diff()`). Then count distinct transactions per day and find the day with the fewest in `low_day`, with its count in `low_n`.

In [ ]:
# Your turn: dup = log.duplicated('txn_id')   (True for the second and later copies)
# Your turn: n_dup = how many rows are duplicates
# Your turn: gap_hours = the largest difference between sorted times, in hours, rounded to 1 decimal
# Your turn: per_day = rows per day among the non duplicates (group by log.ts.dt.date)
# Your turn: low_day (as text) and low_n for the day with the fewest rows


In [ ]:
_missing = [v for v in ('gap_hours', 'low_day', 'low_n', 'n_dup',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert n_dup == 300, 'There are 300 repeated txn_id rows'
assert gap_hours == 6.1, 'The longest silence is 6.1 hours'
assert low_day == '2026-09-17' and low_n == 389, 'The quietest day is 17 September with 389 rows'
print('Task 2 correct')

## Task 3. A schema alert
Write `check_schema(df)`. It raises `ValueError` with the text `ALERT schema | platform lead | <name> is missing` when an expected column is absent. Test it on `raw.rename(columns={'agent_id': 'agent'})` and store the alert text in `msg`.

In [ ]:
WANT = ['txn_id', 'timestamp', 'msisdn_hash', 'agent_id', 'amount_ugx', 'type', 'status']

def check_schema(df):
    # Your turn: gone = the names in WANT that are not in df.columns
    # Your turn: if gone, raise ValueError with the alert text from the task
    pass

check_schema(raw)
# Your turn: call check_schema on the renamed table inside try/except ValueError and keep the text in msg
msg = ''


In [ ]:
_missing = [v for v in ('msg',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert msg == 'ALERT schema | platform lead | agent_id is missing', 'The alert should name the kind, the owner and the missing column exactly as in the task'
print('Task 3 correct')

## Task 4. Publish twice
Write `publish(clean, folder, keyed)` that adds the new rows to `clean.csv` in the folder, sorts by `txn_id`, and when `keyed` is True drops repeated `txn_id` (keep the last). It returns the number of rows in the file. Run it twice on `clean = log[~dup]` for each setting. Store the two row counts in `keyed_rows` and `unkeyed_rows`.

In [ ]:
clean = log[~dup]

def publish(clean, folder, keyed=True):
    f = pathlib.Path(folder) / 'clean.csv'
    # Your turn: if f exists, read it (dtype=str) and pd.concat it with clean
    # Your turn: if keyed, drop_duplicates('txn_id', keep='last')
    # Your turn: write clean sorted by txn_id to f without the index, and return len(clean)
    pass

# Your turn: run publish twice into one new folder (tempfile.mkdtemp()) with keyed=True: keyed_rows = [first, second]
# Your turn: the same with keyed=False: unkeyed_rows


In [ ]:
_missing = [v for v in ('keyed_rows', 'unkeyed_rows',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert keyed_rows == [19700, 19700], 'A keyed load should give 19700 rows after both runs'
assert unkeyed_rows == [19700, 39400], 'Without a key the second run doubles the rows to 39400'
print('Task 4 correct')

## Task 5. The zones in SQL
Build three tables in an in-memory SQLite database: `raw_mm` (all rows), `clean_mm` (first row per `txn_id`) and `serving_mm` (successful transactions with a positive amount, one row per type with `txns` and `value_ugx`). Store the row counts of the first two in `n_raw` and `n_clean`, and the number of cash_out transactions in `n_cash_out`.

In [ ]:
con = sqlite3.connect(':memory:')
raw.assign(amount_ugx=raw.amount_ugx.astype(int)).to_sql('raw_mm', con, index=False)
# Your turn: con.execute('CREATE TABLE clean_mm AS SELECT ... FROM raw_mm WHERE rowid IN (SELECT MIN(rowid) ... GROUP BY txn_id)')
# Your turn: con.execute("CREATE TABLE serving_mm AS SELECT type, COUNT(*) AS txns, SUM(amount_ugx) AS value_ugx FROM clean_mm WHERE ... GROUP BY type")
# Your turn: n_raw, n_clean and n_cash_out from SELECT queries (fetchone()[0])


In [ ]:
_missing = [v for v in ('n_cash_out', 'n_clean', 'n_raw',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert n_raw == 20000 and n_clean == 19700, 'raw_mm holds 20000 rows and clean_mm 19700'
assert n_cash_out == 6542, 'Successful cash_out transactions with a positive amount: 6542'
print('Task 5 correct')

## For the leader
Look at what the pipeline caught: 300 duplicates, a 6.1 hour silence and a renamed column that stops the run. For your most important feed, who is woken up at night when it goes quiet, who is told the next morning, and what delay will your business accept? Write the names (roles) and the numbers below.